# Export Stage 1 (GANomaly) -> ONNX + INT8 for Raspberry Pi

Converts the trained Stage 1 GANomaly generator into an **INT8-quantized ONNX** model for CPU-only deployment (Raspberry Pi 5), and verifies it behaves like the FP32 original before anything ships.

**Before running, attach these inputs (right panel -> Add Input):**
1. **Your Work** -> the saved version of the Stage 1 GANomaly notebook (provides `ganomaly_simple/checkpoints/ganomaly_simple_best.pt` and `ganomaly_simple/stage1_config.json`).
2. The same three datasets used for training (IDRiD, ODIR-5K, APTOS) - needed for quantization calibration and the parity check.

**What this notebook produces** (in `/kaggle/working/deploy/`):
- `ganomaly_int8.onnx` - the INT8 generator (input 1x3x128x128, outputs `xhat, z, zhat`)
- `stage1_deploy.json` - threshold + z-normalisation parameters the device needs to compute anomaly scores

**Gates that must pass before download:** ONNX checker, FP32-ONNX vs PyTorch allclose, and the INT8 parity check (score correlation, AUC drift, gate-decision match).

In [ ]:
# Setup
import os, glob, random, math, json, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn

print("Torch:", torch.__version__)

# --- locate the attached Stage 1 notebook output ---
def find_in_inputs(name):
    hits = [p for p in Path("/kaggle/input").rglob(name) if p.is_file()]
    return hits[0] if hits else None

CKPT = find_in_inputs("ganomaly_simple_best.pt")
STAGE1_CONFIG = find_in_inputs("stage1_config.json")
assert CKPT is not None, "ganomaly_simple_best.pt not found - attach the Stage 1 notebook output (Add Input -> Your Work)"
assert STAGE1_CONFIG is not None, "stage1_config.json not found - attach the Stage 1 notebook output"
print("Checkpoint:    ", CKPT)
print("Stage 1 config:", STAGE1_CONFIG)

# --- dataset paths (same as training) ---
IDRID_ROOT = Path("/kaggle/input/datasets/lakshmiprathik/idrid-516/IDRiD")
ODIR_ROOT = Path("/kaggle/input/datasets/lakshmiprathik/odir-5k/ODIR-5K")
APTOS_ROOT = Path("/kaggle/input/datasets/mariaherrerot/aptos2019")
APTOS_CSV = APTOS_ROOT / "train_1.csv"
APTOS_IMAGE_DIR = (APTOS_ROOT / "train_images" / "train_images") if (APTOS_ROOT / "train_images" / "train_images").exists() else (APTOS_ROOT / "train_images")

WORK = Path("/kaggle/working/stage1_export")
CACHE = WORK / "cache"
DEPLOY = Path("/kaggle/working/deploy")
for d in (WORK, CACHE, DEPLOY):
    d.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 128          # Stage 1 operates at 128x128
LATENT = 128
SEED = 42
N_QUANT_CAL = 200       # calibration images for INT8 static quantization
DEVICE = "cpu"          # export + quantization are CPU work

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
print("IDRiD:", IDRID_ROOT.exists(), "| ODIR:", ODIR_ROOT.exists(), "| APTOS:", APTOS_ROOT.exists())

## Rebuild the dataset index and splits

Byte-for-byte the same logic as the Stage 1 training notebook (same seed), so the calibration/test partitions - and therefore the z-normalisation parameters - are identical to the run that produced threshold 0.6282.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def all_images(root):
    return [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]

def idrid_rows():
    rows = []
    for split in ["train", "validation", "test"]:
        for grade in range(5):
            folder = IDRID_ROOT / split / str(grade)
            if not folder.exists():
                continue
            for image_path in all_images(folder):
                rows.append({"path": str(image_path), "grade": grade, "source": "idrid", "split": split})
    return rows

def aptos_rows():
    aptos_df = pd.read_csv(APTOS_CSV)
    rows = []
    for row in aptos_df.itertuples(index=False):
        image_id = str(row.id_code)
        candidates = [APTOS_IMAGE_DIR / f"{image_id}.png",
                      APTOS_IMAGE_DIR / f"{image_id}.jpg",
                      APTOS_IMAGE_DIR / f"{image_id}.jpeg"]
        image_path = next((p for p in candidates if p.exists()), None)
        if image_path is not None:
            rows.append({"path": str(image_path), "grade": int(row.diagnosis), "source": "aptos", "split": "all"})
    return rows

def odir_rows():
    return [{"path": str(p), "grade": 0, "source": "odir", "split": "all"} for p in all_images(ODIR_ROOT)]

rows = idrid_rows() + aptos_rows() + odir_rows()
df = pd.DataFrame(rows).drop_duplicates(subset="path").reset_index(drop=True)
df["label"] = (df["grade"] > 0).astype(int)
print("Total unique images:", len(df))

def crop_square(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > 7
    if mask.sum() > 100:
        ys, xs = np.where(mask)
        img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    h, w = img.shape[:2]
    s = max(h, w)
    canvas = np.zeros((s, s, 3), np.uint8)
    y = (s - h)//2; x = (s - w)//2
    canvas[y:y+h, x:x+w] = img
    return canvas

def preprocess(path, size=IMG_SIZE):
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((size, size, 3), np.uint8)
    img = crop_square(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

def save_cache(frame):
    cached = []
    for i, r in tqdm(frame.iterrows(), total=len(frame), desc="preprocessing"):
        out = CACHE / (str(i) + ".png")
        if not out.exists():
            cv2.imwrite(str(out), preprocess(r.path))
        cached.append(str(out))
    frame = frame.copy()
    frame["cache_path"] = cached
    return frame

df = save_cache(df)

# --- identical split logic to the Stage 1 notebook (same SEED) ---
def make_splits(frame):
    train_healthy = frame[(frame.label == 0) & (frame.source.isin(["idrid", "aptos", "odir"]))].sample(frac=1, random_state=SEED)
    n = len(train_healthy)
    ncal = max(50, int(.15 * n))
    ntest = max(100, int(.20 * n))
    calib_h = train_healthy.iloc[:ncal]
    test_h = train_healthy.iloc[ncal:ncal+ntest]
    train_h = train_healthy.iloc[ncal+ntest:]
    abnormal = frame[frame.label == 1].sample(frac=1, random_state=SEED)
    ncal_p = min(max(50, int(.20 * len(abnormal))), len(abnormal) // 2)
    calib_p = abnormal.iloc[:ncal_p]
    test_p = abnormal.iloc[ncal_p:]
    return {"train": train_h.reset_index(drop=True),
            "calib": pd.concat([calib_h, calib_p]).sample(frac=1, random_state=1).reset_index(drop=True),
            "test": pd.concat([test_h, test_p]).sample(frac=1, random_state=2).reset_index(drop=True)}

splits = make_splits(df)
for k, v in splits.items():
    print(k, len(v), "healthy=", int((v.grade == 0).sum()), "abnormal=", int((v.grade > 0).sum()))

## Load the trained generator and export ONNX

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent=LATENT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.LeakyReLU(.2, True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(.2, True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(.2, True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(.2, True),
            nn.Conv2d(256, latent, 4, 1, 0))
    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self, latent=LATENT):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent, 256, 4, 1, 0), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh())
    def forward(self, z):
        return self.net(z)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = Encoder(); self.g = Decoder(); self.e2 = Encoder()
    def forward(self, x):
        z = self.e1(x); xhat = self.g(z); zhat = self.e2(xhat)
        return xhat, z, zhat

G = Generator()
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
G.load_state_dict(ckpt["G"])
G.eval()
print(f"Loaded generator from epoch {ckpt['epoch']} (lossG={ckpt['lossG']:.5f})")

# --- ONNX export ---
import onnx
import onnxruntime as ort

ONNX_FP32 = WORK / "ganomaly_fp32.onnx"
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    G, dummy, str(ONNX_FP32),
    input_names=["input"], output_names=["xhat", "z", "zhat"],
    opset_version=17, dynamo=False,
)
onnx.checker.check_model(str(ONNX_FP32))
print("ONNX exported and valid:", ONNX_FP32, f"({ONNX_FP32.stat().st_size / 1e6:.1f} MB)")

# --- FP32 smoke test: PyTorch vs ONNX Runtime must agree ---
sess = ort.InferenceSession(str(ONNX_FP32), providers=["CPUExecutionProvider"])
x_np = np.random.RandomState(0).randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
with torch.no_grad():
    t_out = G(torch.from_numpy(x_np))
o_out = sess.run(None, {"input": x_np})
for name, t, o in zip(["xhat", "z", "zhat"], t_out, o_out):
    ok = np.allclose(t.numpy(), o, atol=1e-4)
    print(f"  {name}: max abs diff {np.abs(t.numpy() - o).max():.2e} -> {'OK' if ok else 'MISMATCH'}")
    assert ok, f"FP32 ONNX mismatch on {name}"
print("FP32 ONNX matches PyTorch.")

## INT8 static quantization

Static post-training quantization (QDQ format, per-channel weights) calibrated on preprocessed **healthy training images** - the same distribution the gate sees in production. Calibration inputs use the exact training normalisation (`x/127.5 - 1`).

In [ ]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

input_name = ort.InferenceSession(str(ONNX_FP32), providers=["CPUExecutionProvider"]).get_inputs()[0].name
print("ONNX input name:", input_name)

def load_batch(paths, batch_size=16):
    imgs = []
    for p in paths:
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        x = img.astype(np.float32) / 127.5 - 1.0
        imgs.append(x.transpose(2, 0, 1))
        if len(imgs) == batch_size:
            yield np.stack(imgs).astype(np.float32)
            imgs = []
    if imgs:
        yield np.stack(imgs).astype(np.float32)

class FundusCalibrationReader(CalibrationDataReader):
    def __init__(self, paths, in_name, batch_size=16):
        self.gen = load_batch(paths, batch_size)
        self.in_name = in_name
    def get_next(self):
        try:
            return {self.in_name: next(self.gen)}
        except StopIteration:
            return None

cal_paths = splits["train"].sample(min(N_QUANT_CAL, len(splits["train"])), random_state=SEED)["cache_path"].tolist()
ONNX_INT8 = WORK / "ganomaly_int8.onnx"
t0 = time.time()
quantize_static(
    model_input=str(ONNX_FP32),
    model_output=str(ONNX_INT8),
    calibration_data_reader=FundusCalibrationReader(cal_paths, input_name),
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QInt8,
    weight_type=QuantType.QInt8,
    per_channel=True,
)
print(f"Quantized in {time.time() - t0:.0f}s")
print(f"FP32: {ONNX_FP32.stat().st_size / 1e6:.1f} MB -> INT8: {ONNX_INT8.stat().st_size / 1e6:.1f} MB")
onnx.checker.check_model(str(ONNX_INT8))
print("INT8 ONNX valid.")

## Parity gate: INT8 must behave like FP32

Scores are computed exactly as in training (`0.5 * latent z + 0.5 * top-1% residual z`, normalisation fitted on healthy calibration images). The gate passes only if INT8 and FP32 scores correlate at > 0.99, test AUC drifts < 0.01, and the gate decision at the deployed threshold matches on >= 99% of images.

In [ ]:
sess_f32 = ort.InferenceSession(str(ONNX_FP32), providers=["CPUExecutionProvider"])
sess_i8 = ort.InferenceSession(str(ONNX_INT8), providers=["CPUExecutionProvider"])

def score_frames(sess, frame, batch_size=32, desc="scoring"):
    rows = []
    for xb in tqdm(list(load_batch(frame["cache_path"].tolist(), batch_size)), desc=desc):
        xhat, z, zhat = sess.run(None, {input_name: xb})
        latent = ((z - zhat) ** 2).mean(axis=(1, 2, 3))
        residual = np.abs(xb - xhat).mean(axis=1)
        flat = residual.reshape(residual.shape[0], -1)
        k = max(1, int(.01 * flat.shape[1]))
        top = np.sort(flat, axis=1)[:, -k:].mean(axis=1)
        mean = flat.mean(axis=1)
        for l, t, m in zip(latent, top, mean):
            rows.append({"latent": float(l), "top_residual": float(t), "mean_residual": float(m)})
    out = pd.DataFrame(rows)
    out["label"] = frame["label"].values
    out["grade"] = frame["grade"].values
    return out

# z-normalisation params from healthy calibration images (identical formula to training)
def fit_params(calib_raw):
    healthy = calib_raw[calib_raw.label == 0]
    params = {}
    for c in ["latent", "top_residual", "mean_residual"]:
        med = healthy[c].median()
        scale = healthy[c].quantile(.75) - healthy[c].quantile(.25) + 1e-8
        params[c] = (float(med), float(scale))
    return params

def apply_scores(raw, params):
    for c, (med, scale) in params.items():
        raw[c + "_z"] = (raw[c] - med) / scale
    raw["score"] = .5 * raw.latent_z + .5 * raw.top_residual_z
    return raw

from sklearn.metrics import roc_auc_score

# fit params on the FP32 calib raw scores, then reuse for everything
calib_raw_f32 = score_frames(sess_f32, splits["calib"], desc="fp32 calib")
params = fit_params(calib_raw_f32)
calib_f32 = apply_scores(calib_raw_f32.copy(), params)
test_f32 = apply_scores(score_frames(sess_f32, splits["test"], desc="fp32 test").copy(), params)

calib_i8 = apply_scores(score_frames(sess_i8, splits["calib"], desc="int8 calib").copy(), params)
test_i8 = apply_scores(score_frames(sess_i8, splits["test"], desc="int8 test").copy(), params)

threshold = json.load(open(STAGE1_CONFIG))["threshold_youden"]
auc_f32 = roc_auc_score(test_f32.label, test_f32.score)
auc_i8 = roc_auc_score(test_i8.label, test_i8.score)
corr = np.corrcoef(test_f32.score, test_i8.score)[0, 1]
match = ((test_f32.score > threshold) == (test_i8.score > threshold)).mean()

print(f"Deployed threshold: {threshold:.4f}")
print(f"Test AUC  FP32: {auc_f32:.4f} | INT8: {auc_i8:.4f} | drift: {abs(auc_f32 - auc_i8):.4f}")
print(f"Score correlation (FP32 vs INT8): {corr:.5f}")
print(f"Gate-decision match at threshold: {match:.4%}")

gates = {
    "score correlation > 0.99": corr > 0.99,
    "AUC drift < 0.01": abs(auc_f32 - auc_i8) < 0.01,
    "decision match >= 99%": match >= 0.99,
}
for g, ok in gates.items():
    print(("PASS " if ok else "FAIL ") + g)
print()
print("OVERALL:", "PASS - safe to deploy" if all(gates.values()) else "FAIL - do NOT deploy; investigate quantization first")

## Benchmark + write deployment artifacts

In [ ]:
# CPU latency (Kaggle CPU is only a rough proxy - benchmark.py gives true numbers on the Pi)
def bench(sess, n=50):
    xb = next(load_batch(cal_paths[:1], 1))
    for _ in range(5):
        sess.run(None, {input_name: xb})          # warmup
    t0 = time.perf_counter()
    for _ in range(n):
        sess.run(None, {input_name: xb})
    return (time.perf_counter() - t0) / n * 1000

print(f"FP32: {bench(sess_f32):.1f} ms/image | INT8: {bench(sess_i8):.1f} ms/image (Kaggle CPU, single thread default)")

deploy_cfg = {
    "model": "ganomaly_simple (Encoder-Decoder-Encoder), INT8 ONNX",
    "img_size": IMG_SIZE,
    "input_name": input_name,
    "output_names": ["xhat", "z", "zhat"],
    "preprocessing": "crop + CLAHE + resize 128, then x/127.5 - 1",
    "score": "0.5 * latent_z + 0.5 * top_residual_z (top-1% residual)",
    "threshold_youden": threshold,
    "znorm_params": {c: {"median": m, "scale": s} for c, (m, s) in params.items()},
    "fp32_test_auc": float(auc_f32),
    "int8_test_auc": float(auc_i8),
    "source_notebook": "Stage 1 GANomaly (notebook834f1c81b4)",
}
with open(DEPLOY / "stage1_deploy.json", "w") as f:
    json.dump(deploy_cfg, f, indent=2)
shutil.copy(ONNX_INT8, DEPLOY / "ganomaly_int8.onnx")

print()
print("=== DOWNLOAD THESE TWO FILES (right panel -> Output -> deploy/) ===")
print("  deploy/ganomaly_int8.onnx")
print("  deploy/stage1_deploy.json")
print()
print("Then place them in deploy/raspberry_pi/models/ on your laptop / Raspberry Pi.")